# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmed-khaled123/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring (Lane 2).**

This is the lane the starter playground itself already builds end to end (`scripts/01`-`05`), so I have a
working baseline, a trained model, and verified evaluation numbers to build on for the next 7 weeks instead
of starting from zero in Week 5. The core question fits how FlyRank's editors actually work: they have
limited review capacity every sprint and need a ranked list of *which pages to look at first*, not a single
global "the SEO is bad" verdict. Section 3 below backs this choice with real numbers from the starter data:
a large share of pages already show the "declining with demand" and "low-CTR-but-visible" patterns this lane
targets, and the starter model already beats the hand-written rule by a wide margin on Precision@50 -- so
there is real signal here worth 7 more weeks of work, not just noise.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

**Research question:** Given a client's content inventory in a review window, which pages should a content
editor review first for refresh, expansion, or protection?

**Decision it improves:** which pages enter this sprint's limited editorial review queue, ranked instead of
picked at random or by gut feel.

**Who acts, and what they do:** a content/SEO editor with fixed weekly capacity (the starter pipeline assumes
roughly the top 50 candidates per review cycle). They open the top of the ranked queue, read the reason
codes, and decide whether to refresh, expand, protect, or leave a page alone.

**Cost of a wrong call, both directions:**
- **False positive** (page ranked high but wasn't actually worth reviewing): wastes an editor's limited hours
  on a page that didn't need attention -- the scarce resource here is editor time, not compute.
- **False negative** (a genuinely declining, high-demand page never surfaces near the top): the page keeps
  losing visibility and clicks silently until the next review cycle, which is a real, measurable traffic cost
  since it already has demand (people are searching for it).

**Why data or ML can help at all:** a single hand-written rule (e.g. "stale AND visible") only fires on 0.1%
of the starter pages (see Section 3) -- it is too narrow to rank a whole inventory. The signal that actually
separates "worth reviewing" from "fine as is" is spread across many correlated, non-linear signals (traffic,
position, freshness, CTR, content depth) that shift per client, which is exactly the kind of messy-but-real
pattern a simple model can learn and a human cannot hand-write as one rule.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
import pandas as pd, json

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
n = len(df)

# Reason-code style checks straight from the lane guide / starter baseline
stale_visible = df[(df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)]
declining_with_demand = df[(df["trend_direction"].str.lower() == "down") & (df["impressions_90d"] >= 100)]
low_ctr_visible = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0)
                      & (df["avg_position"] <= 20) & (df["ctr"] < 0.5)]

print(f"Rows: {n:,} across {df['client_id'].nunique()} clients")
print(f"1) stale_visible_page:      {len(stale_visible):>6,}  ({100*len(stale_visible)/n:.1f}% of pages)")
print(f"2) declining_with_demand:   {len(declining_with_demand):>6,}  ({100*len(declining_with_demand)/n:.1f}% of pages)")
print(f"3) low_ctr_visible_page:    {len(low_ctr_visible):>6,}  ({100*len(low_ctr_visible)/n:.1f}% of pages)")

# Does a learned ranking actually beat the hand-written rule on this lane's own metric?
res = json.load(open("outputs/model_results.json"))
base_p50 = res["baseline"]["baseline_precision_at_50"]
rf_p50 = res["models"]["random_forest"]["precision_at_50"]
print(f"\n4) Baseline rule  Precision@50: {base_p50:.3f}  (~{round(base_p50*50)} of the top 50 right)")
print(f"   Random forest   Precision@50: {rf_p50:.3f}  (~{round(rf_p50*50)} of the top 50 right)")
print(f"   -> learned ranking beats the hand rule by {rf_p50/base_p50:.1f}x on this metric (client-holdout validated).")

print("\nWhy this backs Lane 2: 43.8% of pages already show real, measurable demand behind a decline, and")
print("nearly a third are visible pages under-capturing clicks for their position -- both are large enough")
print("pools to rank meaningfully. And a single hand rule barely covers this ground (0.1% stale-visible hits),")
print("while a learned model already outperforms the rule 3x on the metric this lane is judged by.")


Rows: 30,000 across 32 clients
1) stale_visible_page:          17  (0.1% of pages)
2) declining_with_demand:   13,152  (43.8% of pages)
3) low_ctr_visible_page:     9,759  (32.5% of pages)

4) Baseline rule  Precision@50: 0.240  (~12 of the top 50 right)
   Random forest   Precision@50: 0.740  (~37 of the top 50 right)
   -> learned ranking beats the hand rule by 3.1x on this metric (client-holdout validated).

Why this backs Lane 2: 43.8% of pages already show real, measurable demand behind a decline, and
nearly a third are visible pages under-capturing clicks for their position -- both are large enough
pools to rank meaningfully. And a single hand rule barely covers this ground (0.1% stale-visible hits),
while a learned model already outperforms the rule 3x on the metric this lane is judged by.


## 4. Careful words: what I can and can't claim

**What this work CAN say, by the end of 7 weeks:**
- Observed, directional patterns: "pages with X and Y tend to show Z in this dataset."
- Decision-support rankings: "of the pages I could review this sprint, these are the ones the evidence points
  to first" -- a prioritized list backed by reason codes a human can inspect and override.
- Comparative, honestly-validated numbers: how a learned ranking compares to a transparent hand-written rule,
  measured with client-holdout validation (whole clients kept out of training) so the comparison isn't
  flattered by memorizing a client's quirks.

**What this work will NEVER claim:**
- That refreshing a page *caused* a recovery -- that needs a controlled experiment (e.g. before/after with a
  held-out control group), which this dataset does not provide.
- That any result reveals a Google ranking-algorithm factor, or predicts an AI platform's citation behavior.
- That a high score *guarantees* a page is actually declining -- `trend_direction` in the starter data is a
  proxy computed from the current window, not a confirmed future outcome, so early results are treated as
  "worth a human look," not verdicts.
- That a pattern found in this 30,000-row / 32-client starter slice automatically holds across FlyRank's full
  ~79M-row warehouse without re-checking there.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.